# Student Dropout Early Warning System
## Corrected Preprocessing Pipeline

**Pipeline Overview:**
1. Load raw data
2. Handle the Target correctly (binary: Dropout vs Graduate)
3. Separate Enrolled students (inference set) from training data
4. One-Hot Encode categorical columns properly
5. Scale continuous columns
6. Train/Test split
7. Apply SMOTE only on training data
8. Save everything cleanly for modeling

## Step 0 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

print('All imports successful')

## Step 1 — Load Raw Data
We load the **original** dataset, not your previously processed one.
We want to redo preprocessing correctly from scratch.

In [ ]:
df = pd.read_csv('dataset.csv')

print(f'Shape: {df.shape}')
print(f'\nTarget distribution:')
print(df['Target'].value_counts())
print(f'\nNull values: {df.isnull().sum().sum()}')

df.head()

## Step 2 — Define Column Types

We split columns into three groups:
- **Binary columns** — already 0/1, no encoding needed
- **Nominal categorical** — codes that have no order (e.g. Course ID, Nationality) → need One-Hot Encoding
- **Continuous** — numerical values → need StandardScaler

> **Why no outlier removal?**  
> A 70-year-old enrolling is a real data point, not an error.  
> Removing outliers in behavioral/human data loses real signal.  
> We skip IQR-based removal here.

In [ ]:
# These are already 0/1 — leave them as-is
binary_cols = [
    'Daytime/evening attendance',
    'Displaced',
    'Educational special needs',
    'Debtor',
    'Tuition fees up to date',
    'Gender',
    'Scholarship holder',
    'International',
]

# These are integer codes with NO natural order — must be One-Hot Encoded
# (e.g. Course 3 is not "more" than Course 1, they're just different courses)
nominal_cols = [
    'Marital status',
    'Application mode',
    'Application order',
    'Course',
    'Previous qualification',
    'Nacionality',
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
]

# True numerical values — apply StandardScaler
continuous_cols = [
    'Age at enrollment',
    'Curricular units 1st sem (credited)',
    'Curricular units 1st sem (enrolled)',
    'Curricular units 1st sem (evaluations)',
    'Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)',
    'Curricular units 1st sem (without evaluations)',
    'Curricular units 2nd sem (credited)',
    'Curricular units 2nd sem (enrolled)',
    'Curricular units 2nd sem (evaluations)',
    'Curricular units 2nd sem (approved)',
    'Curricular units 2nd sem (grade)',
    'Curricular units 2nd sem (without evaluations)',
    'Unemployment rate',
    'Inflation rate',
    'GDP',
]

print(f'Binary cols: {len(binary_cols)}')
print(f'Nominal cols to OHE: {len(nominal_cols)}')
print(f'Continuous cols to scale: {len(continuous_cols)}')

# Quick sanity check — make sure we haven't missed or double-counted any column
all_feature_cols = binary_cols + nominal_cols + continuous_cols
original_feature_cols = [c for c in df.columns if c != 'Target']
missed = set(original_feature_cols) - set(all_feature_cols)
print(f'\nMissed columns (should be empty): {missed}')

## Step 3 — Split Enrolled Students Out (Critical Step)

**Why we do this:**  
Our goal is an *early warning system* — we want to predict dropout for currently enrolled students.  
But we can only learn patterns from students whose outcome we already know (Graduated or Dropped Out).  

- `df_model` → students with known outcomes → used for training and testing
- `df_enrolled` → currently enrolled students → this is what we'll predict on later

Enrolled students **never** touch the training or test process.

In [ ]:
# Separate the enrolled students out before any further processing
df_enrolled = df[df['Target'] == 'Enrolled'].copy()
df_model = df[df['Target'] != 'Enrolled'].copy()

print(f'Total rows: {len(df)}')
print(f'Enrolled (inference set, set aside): {len(df_enrolled)}')
print(f'Known outcome rows (for training): {len(df_model)}')
print(f'\nClass distribution in training pool:')
print(df_model['Target'].value_counts())

## Step 4 — Create Binary Target

Now that we only have Graduated and Dropout students,  
we create a simple binary target:  
- `1` = Dropout (the at-risk class we want to catch)
- `0` = Graduate

In [ ]:
# Create binary target
df_model['target_binary'] = (df_model['Target'] == 'Dropout').astype(int)

print('Target encoding:')
print(df_model.groupby('Target')['target_binary'].first())
print(f'\nClass balance:')
print(df_model['target_binary'].value_counts())
print(f'\nDropout rate: {df_model["target_binary"].mean():.1%}')

# Drop the original Target string column
df_model = df_model.drop(columns=['Target'])

## Step 5 — One-Hot Encode Nominal Columns

**Why this matters:**  
Columns like `Course` (values: 1, 2, 3... 17) look numeric but are just ID codes.  
If we leave them as integers, the model thinks Course 17 is 17x more important than Course 1.  
OHE turns each category into its own 0/1 column.

We use `drop='first'` to avoid the dummy variable trap  
(prevents perfect multicollinearity in Logistic Regression).

In [ ]:
# One-Hot Encode nominal columns
# drop_first=True avoids the dummy variable trap
df_encoded = pd.get_dummies(df_model, columns=nominal_cols, drop_first=True)

# Do the same for enrolled students (so they match the same columns later)
df_enrolled_processed = df_enrolled.drop(columns=['Target'])
df_enrolled_encoded = pd.get_dummies(df_enrolled_processed, columns=nominal_cols, drop_first=True)

print(f'Shape before OHE: {df_model.shape}')
print(f'Shape after OHE:  {df_encoded.shape}')
print(f'New columns added: {df_encoded.shape[1] - df_model.shape[1]}')

## Step 6 — Separate X and y, Then Train/Test Split

We split BEFORE scaling.  
The scaler should only learn statistics (mean, std) from training data.  
If we scale first, test data info leaks into training — that's a subtle form of data leakage.

In [ ]:
# Separate features and target
X = df_encoded.drop(columns=['target_binary'])
y = df_encoded['target_binary']

# stratify=y ensures both train and test sets have similar class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y  # Important for imbalanced classes
)

print(f'Training set:   {X_train.shape}')
print(f'Test set:       {X_test.shape}')
print(f'\nClass balance in training set:')
print(y_train.value_counts())
print(f'\nClass balance in test set:')
print(y_test.value_counts())

## Step 7 — Scale Continuous Columns

**Key rule:** `fit_transform` on training data only.  
Only `transform` (not fit) on test data and enrolled data.  

This way the scaler never "sees" any test or inference data.

In [ ]:
# Only scale the continuous columns that exist in X after OHE
# (some continuous cols might be the same, OHE didn't touch them)
cols_to_scale = [c for c in continuous_cols if c in X_train.columns]

scaler = StandardScaler()

# Fit ONLY on training data, then transform
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])

# Only transform test data — never fit on it
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

print(f'Scaled {len(cols_to_scale)} continuous columns')
print(f'\nSample of scaled values (should be roughly -3 to +3):')
X_train[cols_to_scale[:3]].describe().round(3)

## Step 8 — Apply SMOTE (Training Data Only!)

**What SMOTE does:**  
Synthetically generates new minority class (Dropout) samples by interpolating  
between existing Dropout students in feature space.

**Critical rule:** Apply SMOTE **only on X_train**.  
Never on X_test — test set must stay real, unmodified data  
so your evaluation metrics reflect real-world performance.

In [ ]:
print('Before SMOTE:')
print(f'  Graduates (0): {(y_train == 0).sum()}')
print(f'  Dropouts  (1): {(y_train == 1).sum()}')

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f'\nAfter SMOTE:')
print(f'  Graduates (0): {(y_train_resampled == 0).sum()}')
print(f'  Dropouts  (1): {(y_train_resampled == 1).sum()}')
print(f'\nX_test is untouched: {X_test.shape} rows — GOOD')

## Step 9 — Prepare the Enrolled (Inference) Set

Now we apply the same OHE and scaling to the enrolled students.  
These are the students whose dropout risk we'll actually predict.

In [ ]:
# Align enrolled columns to match training columns exactly
# (Some OHE categories might not appear in the enrolled subset)
X_enrolled = df_enrolled_encoded.reindex(columns=X.columns, fill_value=0)

# Scale continuous columns using the SAME scaler fitted on training data
enrolled_cols_to_scale = [c for c in cols_to_scale if c in X_enrolled.columns]
X_enrolled[enrolled_cols_to_scale] = scaler.transform(X_enrolled[enrolled_cols_to_scale])

print(f'Enrolled inference set shape: {X_enrolled.shape}')
print(f'Training features shape:      {X_train_resampled.shape}')
print(f'\nColumn match: {list(X_enrolled.columns) == list(X_train.columns)}')

## Step 10 — Final Summary & Sanity Check

In [ ]:
print('=' * 50)
print('PREPROCESSING COMPLETE — FINAL SUMMARY')
print('=' * 50)
print(f'\nTraining set (after SMOTE): {X_train_resampled.shape}')
print(f'Test set (real data only):  {X_test.shape}')
print(f'Enrolled inference set:     {X_enrolled.shape}')
print(f'\nFeature count: {X_train_resampled.shape[1]}')
print(f'\nTraining class balance:')
vals, counts = np.unique(y_train_resampled, return_counts=True)
for v, c in zip(vals, counts):
    label = 'Dropout' if v == 1 else 'Graduate'
    print(f'  {label} ({v}): {c}')

print(f'\nNull check — train: {np.isnan(X_train_resampled).sum()}')
print(f'Null check — test:  {X_test.isnull().sum().sum()}')
print('\nReady for modeling!')

In [ ]:
# Visualise class balance before and after SMOTE
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(['Graduate', 'Dropout'], [( y_train == 0).sum(), (y_train == 1).sum()],
            color=['steelblue', 'tomato'])
axes[0].set_title('Before SMOTE (Training Set)')
axes[0].set_ylabel('Count')

axes[1].bar(['Graduate', 'Dropout'],
            [(y_train_resampled == 0).sum(), (y_train_resampled == 1).sum()],
            color=['steelblue', 'tomato'])
axes[1].set_title('After SMOTE (Training Set)')

plt.suptitle('Class Balance Before vs After SMOTE', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 11 — Save Processed Data

Save everything so you can load it directly in your modeling notebook  
without repeating preprocessing every time.

In [ ]:
import pickle

# Save datasets as CSV
X_train_resampled_df = pd.DataFrame(X_train_resampled, columns=X_train.columns)
X_train_resampled_df['target'] = y_train_resampled
X_train_resampled_df.to_csv('train_data.csv', index=False)

X_test_df = X_test.copy()
X_test_df['target'] = y_test.values
X_test_df.to_csv('test_data.csv', index=False)

X_enrolled.to_csv('enrolled_inference.csv', index=False)

# Save the scaler so you can use it consistently in modeling
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Saved:')
print('  train_data.csv        — SMOTE-balanced training data')
print('  test_data.csv         — Real test data (never touch this for training)')
print('  enrolled_inference.csv — Students to predict dropout risk for')
print('  scaler.pkl            — Fitted scaler (reuse in modeling notebook)')

## What's Next — Your Modeling Pipeline

In your next notebook, load the saved files and build your models:

```python
import pandas as pd, pickle

train = pd.read_csv('train_data.csv')
test  = pd.read_csv('test_data.csv')

X_train = train.drop('target', axis=1)
y_train = train['target']
X_test  = test.drop('target', axis=1)
y_test  = test['target']

# Then plug into:
# 1. LogisticRegression  (baseline + interpretability)
# 2. RandomForestClassifier
# 3. SVC (with probability=True for calibrated probs)
# 4. XGBClassifier
# 5. StackingClassifier  (meta-learner)
# 6. SHAP explanations
# 7. DiCE counterfactuals

# Evaluate using: classification_report, roc_auc_score, calibration_curve
# Focus on RECALL for Dropout class — missing an at-risk student is costly
```